In [22]:
! pip install langchain langchain-text-splitters langchain-community bs4
! pip install -U "langchain[openai]"
! pip install -U "langchain-openai"
! pip install -U "langchain-core"
! pip install sentence-transformers
! pip install python-dotenv
! pip install fastapi uvicorn


   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ---------------------------------------- 0/3 [uvicorn]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------

### Importation des libs

In [ ]:
import bs4
import os
import getpass

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from dotenv import load_dotenv

load_dotenv()

# API KEY
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API key: ")

# MODEL (OpenRouter)
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# EMBEDDINGS (local, pas OpenAI)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = InMemoryVectorStore(embeddings)

# LOAD
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

# SPLIT
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
all_splits = text_splitter.split_documents(docs)

# STORE
vector_store.add_documents(all_splits)

# TOOL
@tool
def retrieve_context(query: str):
    "retrieve function"
    docs = vector_store.similarity_search(query, k=2)
    return "\n\n".join(doc.page_content for doc in docs)

tools = [retrieve_context]

# AGENT
agent = create_agent(
    model,
    tools,
    system_prompt="Use the tool to answer questions."
)

# RUN
query = "Quel est le CA de Dassaut system en 2025 ?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7987.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


================================ Human Message =================================

Quel est le CA de Dassaut system en 2025 ?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_NIyH09aUzgQ0hal3xMBx98Ab)
 Call ID: call_NIyH09aUzgQ0hal3xMBx98Ab
  Args:
    query: CA de Dassault Systèmes en 2025
================================= Tool Message =================================
Name: retrieve_context

Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this proc